# CS 685/785: Coding Assignment 1

**Spring 2026&mdash;Prof. Brandon Oubre**

In this assignment, you will gain familiarity with the central limit theorem, data visualization, and hypothesis testing.

This assignment is worth a total of 40 points.

## Academic Integrity Declaration

**I declare that:**
- **I have completed this assignment entirely on my own.**
- **I understand and have complied with the course policy on the use of AI tools.**
- **I have read the UAB Academic Integrity Code and understand that any breach of this code may result in severe penalties, including failure of the class.**

Name: <font color='blue'><u>*__You should edit this markdown cell (double-click) to include your name here as an acknowledgement of the academic integrity declaration.__*</u></font>

## Reminder on the Use of AI Tools
For coding assignments, you are permitted limited use of AI tools within the bounds of the policy in the syllabus. In broad strokes, you must:
- Include the prompt you used to generate the code
- Include the original code resulting from the prompt
- Include a citation to the tool used
- Be able to explain any code submitted as part of this assignment
  
<u>You have the ultimate responsibility for the correctness and clarity of any code submitted as part of this assignment.</u> You should thereore understand it, test it, debug it, revise/improve it, and document it.

<u>**You should not use AI to respond to the written prompts asking you to analyze or interpret your results.**</u>

### Example AI Use Documentation

The below cell demonstrates an example of how to document AI use.

---

**AI Model**: Microsoft Copilot with Data Protection (provided by UAB)

**Prompt**:: Write a python program that uses numpy and matplotlib to visualize a Gaussian distribution.

**Result**:
```
import numpy as np
import matplotlib.pyplot as plt

# Generate data for Gaussian distribution
mean = 0
std_dev = 1
num_samples = 1000

data = np.random.normal(mean, std_dev, num_samples)

# Plot the Gaussian distribution
plt.hist(data, bins=30, density=True, alpha=0.6, color='g')

# Plot the probability density function (PDF)
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = np.exp(-0.5*((x - mean) / std_dev)**2) / (std_dev * np.sqrt(2 * np.pi))

plt.plot(x, p, 'k', linewidth=2)
title = "Gaussian Distribution\nMean = {:.2f}, Standard Deviation = {:.2f}".format(mean, std_dev)
plt.title(title)
plt.xlabel('Value')
plt.ylabel('Frequency')

# Show the plot
plt.show()
```

---

## Imports

Do not import additional libraries unless specified by the assignment prompts. When permitted, those imports should not be added here but rather be included as part of your solution.

In [1]:
import os
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

c:\Users\Nick\Documents\Datapacks\Pirates\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading the Data

The data is assumed to be stored in a `data/` folder in the same directory as this notebook.

In [2]:
data_dir = os.path.abspath('data/')
print(f'The data directory is: {data_dir}')

The data directory is: c:\Users\Nick\Downloads\data


In [3]:
file_paths = [
    os.path.join(data_dir, f) for f in os.listdir(data_dir)
    if os.path.splitext(f)[1].lower() == '.xpt'
]
data_files = [pd.read_sas(f) for f in tqdm(file_paths, desc='Loading Data')]

# Merge all the data files into a single DataFrame 
data = data_files[0]
for df in tqdm(data_files[1:], desc='Merging Data'):
    data = pd.merge(data, df, how='outer', on='SEQN', validate='1:1')  # Outer join is important; do not want to drop data missing in one file but not others

print('The data frame has {} rows and {} columns'.format(*data.shape))

if data['SEQN'].duplicated().sum() > 0:
    print('WARNING: The data has duplicated identifiers. Something is wrong.')

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'c:\\Users\\Nick\\Downloads\\data'

## Data Set Documentation

You may find the following documentation links helpful for interpreting the subset of NHANES data used in this assignment:

- [Demographic Variables and Sample Weights (DEMO_L)](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.htm)
- [Body Measures (BMX_L)](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.htm)
- [Cholesterol – High-Density Lipoprotein (HDL_L)](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/HDL_L.htm)
- [Blood Pressure - Oscillometric Measurements (BPXO_L)](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BPXO_L.htm)

## Problem 1

In this problem, we will empirically observe the Central Limit Theorem.

First, we create an `hdl_cholesterol` array that contains the amount of cholesterol (mg/dL) in participants' blood during laboratory tests.

In [ ]:
hdl_cholesterol = data['LBDHDD'].dropna().to_numpy()

### Problem 1.1 [5 points]

Let $X$ be a random variable that takes on the values in `hdl_cholesterol` with equal probability. We will create a new random variable $Y_N=\frac{1}{N}\sum_{i=1}^{N}x_i$ that represents the mean of $N$ samples $x_i$ drawn from $X$.

Create a function (call it `hdl_means`) that takes an argument `n` and returns an array containing **10,000 samples drawn from $Y_n$**.

*Hint: You may use `np.random.choice`. You should think about how to avoid explicitly looping in Python if you want your code to run quickly.*

In [ ]:
def hdl_means(n: int, n_samples: int = 10000) -> np.ndarray:
    """
    Generate samples from the distribution of sample means Y_N.
    
    Parameters:
    -----------
    n : int
        The number of samples to average for each Y_N
    n_samples : int
        The number of Y_N samples to generate (default: 10000)
    
    Returns:
    --------
    np.ndarray
        Array of shape (n_samples,) containing samples from Y_N
    """
    # Draw n*n_samples values from hdl_cholesterol all at once
    # Shape will be (n_samples, n) after reshaping
    samples = np.random.choice(hdl_cholesterol, size=(n_samples, n), replace=True)
    
    # Compute the mean along axis 1 to get the mean of each set of n samples
    # This gives us n_samples values of Y_N
    y_n_samples = np.mean(samples, axis=1)
    
    return y_n_samples

### Problem 1.2 [5 points]
Create a plot with three subplots on a single row. Each plot should contain a [histogram](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html) that shows the approximate distribution of $Y_N$ (i.e., the result of your `hdl_means` function).
The leftmost plot should display $Y_{10}$, the middle plot should display $Y_{30}$, and the right plot should display $Y_{100}$.
In each histogram, plot a vertical black line at the mean using `axvline`.

Ensure that the histograms are directly comparable (e.g., **same** bins, normalized heights, same y axes). Remember to label your axes. A bin width of 1 should show sufficient detail to compare the distributions.

In [ ]:
# Generate samples for different values of N
y_10 = hdl_means(10)
y_30 = hdl_means(30)
y_100 = hdl_means(100)

# Create figure with three subplots in a single row
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Define common bins for all histograms (bin width of 1)
# We need to determine the range across all three distributions
all_data = np.concatenate([y_10, y_30, y_100])
min_val = np.floor(all_data.min())
max_val = np.ceil(all_data.max())
bins = np.arange(min_val, max_val + 1, 1)  # bin width of 1

# Calculate means once
mean_10 = np.mean(y_10)
mean_30 = np.mean(y_30)
mean_100 = np.mean(y_100)

# Plot Y_10
axes[0].hist(y_10, bins=bins, density=True, alpha=0.7, edgecolor='black')
axes[0].axvline(mean_10, color='black', linestyle='-', linewidth=2, label=f'Mean = {mean_10:.2f}')
axes[0].set_xlabel('HDL Cholesterol (mg/dL)')
axes[0].set_ylabel('Probability Density')
axes[0].set_title('$Y_{10}$ (N=10)')
axes[0].legend()

# Plot Y_30
axes[1].hist(y_30, bins=bins, density=True, alpha=0.7, edgecolor='black')
axes[1].axvline(mean_30, color='black', linestyle='-', linewidth=2, label=f'Mean = {mean_30:.2f}')
axes[1].set_xlabel('HDL Cholesterol (mg/dL)')
axes[1].set_ylabel('Probability Density')
axes[1].set_title('$Y_{30}$ (N=30)')
axes[1].legend()

# Plot Y_100
axes[2].hist(y_100, bins=bins, density=True, alpha=0.7, edgecolor='black')
axes[2].axvline(mean_100, color='black', linestyle='-', linewidth=2, label=f'Mean = {mean_100:.2f}')
axes[2].set_xlabel('HDL Cholesterol (mg/dL)')
axes[2].set_ylabel('Probability Density')
axes[2].set_title('$Y_{100}$ (N=100)')
axes[2].legend()

# Set the same y-axis limits for all subplots for direct comparison
y_max = max([ax.get_ylim()[1] for ax in axes])
for ax in axes:
    ax.set_ylim(0, y_max)

plt.tight_layout()
plt.show()

### Problem 1.3 [4 points]
In written language (use a Markdown cell), explain how the Central Limit Theorem predicts what you observed above. (Be sure to consider the distribution, the mean, and the standard deviation.)

<font color='red'>**NO AI USE ALLOWED FOR THIS QUESTION**</font>

<font color="blue">*Your answer here*</font>

### Problem 1.4 [3 points]
For each of $Y_{10}$, $Y_{30}$, and $Y_{100}$, compute the theoretical mean and standard deviation using the Central Limit Theorem. (You may use the empirical value of the mean and standard deviation of $X$, i.e., `hdl_cholesterol`. NumPy has functions for this.)

Then, compute the absolute difference between each theoretical value and the empirically-obtained value (shown by your histograms). Print out each absolute difference in scientific notation. (If your difference is `diff`, then you can format the string as `'{diff:.2e}'`.) Be sure to specify the parameter and distribution corresponding to each difference.

*Hint: You should report six differences, as you are considering two parameters for three distributions.*

In [ ]:
# Compute empirical mean and std for the original distribution X
x_mean = np.mean(hdl_cholesterol)
x_std = np.std(hdl_cholesterol, ddof=1)  # Use sample std

# Generate samples (reuse from 1.2)
y_10 = hdl_means(10)
y_30 = hdl_means(30)
y_100 = hdl_means(100)

# Empirical means and stds
empirical_mean_10 = np.mean(y_10)
empirical_std_10 = np.std(y_10, ddof=1)

empirical_mean_30 = np.mean(y_30)
empirical_std_30 = np.std(y_30, ddof=1)

empirical_mean_100 = np.mean(y_100)
empirical_std_100 = np.std(y_100, ddof=1)

# Theoretical values according to CLT
# Mean of Y_N = Mean of X (same for all N)
theoretical_mean = x_mean

# Std of Y_N = Std of X / sqrt(N)
theoretical_std_10 = x_std / np.sqrt(10)
theoretical_std_30 = x_std / np.sqrt(30)
theoretical_std_100 = x_std / np.sqrt(100)

# Compute absolute differences
diff_mean_10 = abs(empirical_mean_10 - theoretical_mean)
diff_std_10 = abs(empirical_std_10 - theoretical_std_10)

diff_mean_30 = abs(empirical_mean_30 - theoretical_mean)
diff_std_30 = abs(empirical_std_30 - theoretical_std_30)

diff_mean_100 = abs(empirical_mean_100 - theoretical_mean)
diff_std_100 = abs(empirical_std_100 - theoretical_std_100)

# Print results
print("Y_10:")
print(f"  Mean difference: {diff_mean_10:.2e}")
print(f"  Std difference: {diff_std_10:.2e}")
print()
print("Y_30:")
print(f"  Mean difference: {diff_mean_30:.2e}")
print(f"  Std difference: {diff_std_30:.2e}")
print()
print("Y_100:")
print(f"  Mean difference: {diff_mean_100:.2e}")
print(f"  Std difference: {diff_std_100:.2e}")

## Problem 2

In this problem, we will compare the heights of men and women.

To start, we will curate a subset of the data for your use.
The `male` and `female` arrays contain Boolean flags about the sex of each participant. The `height` array contains the height of each participant in centimeters.
The $i^{th}$ entry in each array corresponds to the $i^{th}$ participant.

In [ ]:
prompt2_subset = data[['RIAGENDR', 'RIDAGEYR', 'BMXHT']].dropna()
prompt2_subset = prompt2_subset.loc[prompt2_subset['RIDAGEYR'] >= 18]  # Remove pediatric data

male = (prompt2_subset['RIAGENDR'] == 1).to_numpy()
female = (prompt2_subset['RIAGENDR'] == 2).to_numpy()
height = prompt2_subset['BMXHT'].to_numpy()

### Problem 2.1 [5 points]

Create a single plot with no subplots containing two box plots. One subfigure should show a box plot of heights for men. The other should show a box plot of heights for women.

Remember to label the axis correspond to participant height. The text specifying which group each box plot corresponds to (i.e., the tick labels) should read "Male" and "Female".

In [ ]:
# Extract heights for males and females
male_heights = height[male]
female_heights = height[female]

# Create box plot
plt.figure(figsize=(8, 6))
plt.boxplot([male_heights, female_heights], labels=['Male', 'Female'])
plt.ylabel('Height (cm)')
plt.title('Height Distribution by Sex')
plt.grid(axis='y', alpha=0.3)
plt.show()

### Problem 2.2 [3 points]
You should see a difference in the above plot. Let us now test the hypothesis that a difference actually exists. Use `scipy.stats.ttest_ind` with `equal_var=False` to perform a Welch's $t$-test.

In [ ]:
# Extract heights for males and females
male_heights = height[male]
female_heights = height[female]

# Perform Welch's t-test
t_statistic, p_value = stats.ttest_ind(male_heights, female_heights, equal_var=False)

print(f"t-statistic: {t_statistic:.4f}")
print(f"p-value: {p_value:.4e}")

### Problem 2.3 [3 points]
Assume $\alpha=0.01$. In written language (use a Markdown cell), interpret this result in terms of the null hypothesis and explain your reasoning. What do you ultimately conclude?

<font color='red'>**NO AI USE ALLOWED FOR THIS QUESTION**</font>

<font color="blue">*Your answer here*</font>

## Problem 3
In this question, we will investigate whether there is association between blood pressure and age.

Let us again curate a subset of data for your use. `systolic` and `diastolic` are systolic (during heartbeats) and diastolic (between heartbeats) pressures, respectively. `age` is, of course, participant age.

In [ ]:
prompt3_subset = data[['BPXOSY1', 'BPXODI1', 'RIDAGEYR']].dropna()
systolic = prompt3_subset['BPXOSY1'].to_numpy()
diastolic = prompt3_subset['BPXODI1'].to_numpy()
age = prompt3_subset['RIDAGEYR'].to_numpy()

### Problem 3.1 [5 points]
Create a plot with three subplots in a single row. Each subplot should be a scatter plot between two of the three variables being investigated (systolic pressure, diastolic pressure, and age). Remember to label your axes.

In [ ]:
# Create figure with three subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Subplot 1: Systolic vs Age
axes[0].scatter(age, systolic, alpha=0.3, s=10)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Systolic Pressure (mmHg)')
axes[0].set_title('Systolic Pressure vs Age')
axes[0].grid(alpha=0.3)

# Subplot 2: Diastolic vs Age
axes[1].scatter(age, diastolic, alpha=0.3, s=10)
axes[1].set_xlabel('Age (years)')
axes[1].set_ylabel('Diastolic Pressure (mmHg)')
axes[1].set_title('Diastolic Pressure vs Age')
axes[1].grid(alpha=0.3)

# Subplot 3: Systolic vs Diastolic
axes[2].scatter(diastolic, systolic, alpha=0.3, s=10)
axes[2].set_xlabel('Diastolic Pressure (mmHg)')
axes[2].set_ylabel('Systolic Pressure (mmHg)')
axes[2].set_title('Systolic vs Diastolic Pressure')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Problem 3.2 [3 points]
Use `scipy.stats.pearsonr` too compute the correlation between each pair of variables.

In [ ]:
# Compute Pearson correlation for each pair

# Systolic vs Age
corr_sys_age, p_sys_age = stats.pearsonr(age, systolic)
print("Systolic Pressure vs Age:")
print(f"  Correlation: {corr_sys_age:.4f}")
print(f"  p-value: {p_sys_age:.4e}")
print()

# Diastolic vs Age
corr_dia_age, p_dia_age = stats.pearsonr(age, diastolic)
print("Diastolic Pressure vs Age:")
print(f"  Correlation: {corr_dia_age:.4f}")
print(f"  p-value: {p_dia_age:.4e}")
print()

# Systolic vs Diastolic
corr_sys_dia, p_sys_dia = stats.pearsonr(systolic, diastolic)
print("Systolic vs Diastolic Pressure:")
print(f"  Correlation: {corr_sys_dia:.4f}")
print(f"  p-value: {p_sys_dia:.4e}")

### Problem 3.3 [4 points]
Interpret these correlations in writing (use a Markdown cell). Are there actually associations between each pair of variables? Looking at your scatter plots, explain any differences in correlation observed between systolic pressure and age and between diastolic pressure and age.

<font color='red'>**NO AI USE ALLOWED FOR THIS QUESTION**</font>

<font color="blue">*Your answer here*</font>